In [1]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
from collections import defaultdict

print("Step 1: Multi-Layer Correlation Logic & vSOC Alert Design...")

# MITRE ATT&CK Mapping for Automotive [cite: 92]
MITRE_MAPPING = {
    'DoS': 'T0855: Unauthorized Command Message',
    'Fuzzing': 'T0855: Unauthorized Command Message',
    'Spoofing': 'T0830: Adversary-in-the-Middle',
    'Replay': 'T0843: Replay Attack',
    'Unknown': 'Unmapped Anomaly'
}

class vSOC_Alert_Engine:
    def __init__(self):
        self.can_weight = 0.6
        self.telematics_weight = 0.4
        
    def calculate_confidence(self, can_prob, telematics_prob):
        """Implement confidence scoring based on multiple signal sources """
        # CAN anomaly + unusual telematics = higher confidence 
        return (can_prob * self.can_weight) + (telematics_prob * self.telematics_weight)
    
    def determine_severity(self, confidence_score):
        """Define alert tiers (informational, suspicious, critical) [cite: 91]"""
        if confidence_score >= 0.8:
            return "CRITICAL"
        elif confidence_score >= 0.5:
            return "SUSPICIOUS"
        elif confidence_score > 0.2:
            return "INFORMATIONAL"
        else:
            return "BENIGN"

    def generate_alert(self, vehicle_id, timestamp, can_event, telematics_event):
        """Design alert format suitable for analyst triage [cite: 93]"""
        can_prob = can_event.get('anomaly_probability', 0)
        telematics_prob = telematics_event.get('anomaly_probability', 0)
        attack_type = can_event.get('classified_attack_type', 'Unknown')
        
        confidence = self.calculate_confidence(can_prob, telematics_prob)
        severity = self.determine_severity(confidence)
        
        if severity == "BENIGN":
            return None 
            
        alert = {
            "alert_id": f"VSOC-{np.random.randint(10000, 99999)}",
            "timestamp": timestamp,
            "vehicle_id": vehicle_id,
            "severity": severity,
            "confidence_score": round(confidence, 2),
            "mitre_technique": MITRE_MAPPING.get(attack_type, 'Unmapped Anomaly'),
            "detection_sources": {
                "can_ids_probability": round(can_prob, 2),
                "telematics_behavior_probability": round(telematics_prob, 2)
            },
            "evidence": {
                "can_indicators": can_event.get('indicators', []),
                "telematics_deviations": telematics_event.get('deviations', [])
            }
        }
        return alert

print("\nStep 2: Fleet-Scale Considerations...")

class Fleet_Correlation_Engine:
    def __init__(self, systemic_threshold=3):
        self.active_alerts = []
        self.systemic_threshold = systemic_threshold # Number of unique vehicles to trigger a fleet alert

    def ingest_alert(self, alert):
        """Discuss how to aggregate individual vehicle detections [cite: 95]"""
        if alert and alert['severity'] in ['SUSPICIOUS', 'CRITICAL']:
            self.active_alerts.append(alert)

    def analyze_fleet_status(self):
        """Cross-vehicle correlation for systemic attacks [cite: 96]"""
        # Group alerts by MITRE technique to find coordinated attacks
        technique_to_vins = defaultdict(set)
        
        for alert in self.active_alerts:
            technique = alert['mitre_technique']
            vin = alert['vehicle_id']
            technique_to_vins[technique].add(vin)
            
        fleet_warnings = []
        for technique, vins in technique_to_vins.items():
            if len(vins) >= self.systemic_threshold:
                fleet_warnings.append({
                    "fleet_alert_type": "SYSTEMIC_ATTACK_DETECTED",
                    "mitre_technique": technique,
                    "affected_vehicles_count": len(vins),
                    "affected_vins": list(vins),
                    "recommended_action": "Initiate fleet-wide containment protocols and check OTA update logs for compromise."
                })
        return fleet_warnings

# --- Simulation Execution ---
vsoc_engine = vSOC_Alert_Engine()
fleet_engine = Fleet_Correlation_Engine(systemic_threshold=2) # Set low for simulation purposes

current_time = datetime.utcnow()

# Simulating a coordinated Spoofing attack across multiple vehicles
simulated_events = [
    {"vin": "VIN_001", "can_prob": 0.95, "attack": "Spoofing", "tel_prob": 0.90},
    {"vin": "VIN_002", "can_prob": 0.88, "attack": "Spoofing", "tel_prob": 0.85},
    {"vin": "VIN_003", "can_prob": 0.10, "attack": "Unknown", "tel_prob": 0.05} # Normal vehicle
]

print("Processing individual vehicle events...")
for i, event in enumerate(simulated_events):
    alert = vsoc_engine.generate_alert(
        vehicle_id=event["vin"],
        timestamp=(current_time + timedelta(seconds=i*10)).isoformat() + "Z",
        can_event={"anomaly_probability": event["can_prob"], "classified_attack_type": event["attack"]},
        telematics_event={"anomaly_probability": event["tel_prob"]}
    )
    fleet_engine.ingest_alert(alert)
    if alert:
        print(f"Generated {alert['severity']} alert for {event['vin']} (Confidence: {alert['confidence_score']})")

print("\nExecuting Cross-Vehicle Correlation...")
fleet_alerts = fleet_engine.analyze_fleet_status()

if fleet_alerts:
    print("\n*** FLEET-WIDE INCIDENT DETECTED ***")
    print(json.dumps(fleet_alerts, indent=2))
else:
    print("Fleet status normal. No systemic threats detected.")

Step 1: Multi-Layer Correlation Logic & vSOC Alert Design...

Step 2: Fleet-Scale Considerations...
Processing individual vehicle events...
Generated CRITICAL alert for VIN_001 (Confidence: 0.93)
Generated CRITICAL alert for VIN_002 (Confidence: 0.87)

Executing Cross-Vehicle Correlation...

*** FLEET-WIDE INCIDENT DETECTED ***
[
  {
    "fleet_alert_type": "SYSTEMIC_ATTACK_DETECTED",
    "mitre_technique": "T0830: Adversary-in-the-Middle",
    "affected_vehicles_count": 2,
    "affected_vins": [
      "VIN_001",
      "VIN_002"
    ],
    "recommended_action": "Initiate fleet-wide containment protocols and check OTA update logs for compromise."
  }
]
